# Basketbal Vlaanderen (VBL) ? Foutenanalyse & Technische Fouten per Speler

Dit notebook analyseert wedstrijdverslagen van **Basketbal Vlaanderen (VBL / Wisseq)** voor een gekozen club en ploeg.
Het automatiseert de volledige flow:

1. **Club inladen**: Leest de clubgegevens op basis van de club-GUID (bv. `BVBL1037` voor BBC Haantjes Oudenaarde).
2. **Ploeg selecteren**: Lijst alle ploegen van de club op en laat een specifieke ploeg selecteren (bv. `BVBL1037HSE  1` voor HSE A).
3. **Voorbije wedstrijden (Uitslagen)**: Haalt alle gespeelde wedstrijden op van het huidige seizoen.
4. **Wedstrijdverslagen (DWF)**: Doorloopt elke wedstrijd-GUID (bv. `BVBL26279100LAHSE21AAN`).
5. **Uitgebreid verslag (Matchdetailpdf)**: Vraagt via de VBL WebService de digitale wedstrijdbladen op:
   - Deelnemers / Spelerslijst (`DwfDeelByWedGuid`)
   - Voortgang / Gebeurtenissen (`DwfVgngByWedGuid`)
6. **Fouten per speler per type**: Brengt naast de naam van elke speler alle gemaakte fouten in kaart, uitgesplitst per type:
   - **Persoonlijke fouten** (`P`, `P1`, `P2`, `P3`)
   - **Technische fouten** (`T`, `T1`, `B`, `C`)
   - **Onsportieve fouten** (`U`, `U1`, `U2`)
   - **Diskwalificerende fouten** (`D`, `DI`, `GD`)

In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt

# VBL API Base URL & Headers (publiek toegankelijk, conform Wisseq / VBL DWF)
VBL_BASE_URL = "https://vblcb.wisseq.eu/VBLCB_WebService/data"

API_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "authorization": "na",
    "ssoBearer": "ssoWQAndre",
    "Content-Type": "application/json; charset=utf-8",
}

print("Setup voltooid. Klaar om VBL data op te halen!")

## Stap 1 & 2: Club inladen en ploeg selecteren

We halen alle ploegen van de club op via het `OrgMatchesByGuid` endpoint. We nemen standaard **BBC Haantjes Oudenaarde (`BVBL1037`)** als voorbeeld, maar elke club-GUID kan worden ingegeven.

In [ ]:
CLUB_GUID = "BVBL1037"  # BBC Haantjes Certifisc Oudenaarde

def get_club_teams(club_guid: str) -> pd.DataFrame:
    """Haalt alle unieke ploegen van een club op uit het wedstrijdprogramma."""
    url = f"{VBL_BASE_URL}/OrgMatchesByGuid?issguid={club_guid}"
    resp = requests.get(url, headers=API_HEADERS, timeout=15)
    resp.raise_for_status()
    matches = resp.json()
    
    teams = {}
    for m in matches:
        if club_guid in m.get("tTGUID", ""):
            teams[m.get("tTGUID")] = m.get("tTNaam")
        if club_guid in m.get("tUGUID", ""):
            teams[m.get("tUGUID")] = m.get("tUNaam")
            
    df_teams = pd.DataFrame([
        {"team_guid": guid, "team_naam": naam}
        for guid, naam in sorted(teams.items())
    ])
    return df_teams

df_teams = get_club_teams(CLUB_GUID)
print(f"Gevonden ploegen voor {CLUB_GUID} ({len(df_teams)} ploegen):")
display(df_teams)

### Selecteer de gewenste ploeg
Hieronder kiezen we de ploeg waarvan we de wedstrijden en fouten willen analyseren. Standaard staat dit op **HSE A** (`BVBL1037HSE  1`).

In [ ]:
# Kies hier de gewenste team_guid uit de tabel hierboven:
SELECTED_TEAM_GUID = "BVBL1037HSE  1"  # bv. HSE A

selected_team_name = df_teams.loc[df_teams["team_guid"] == SELECTED_TEAM_GUID, "team_naam"].values[0]
print(f"Geselecteerde ploeg: {selected_team_name} ({SELECTED_TEAM_GUID})")

## Stap 3: Voorbije wedstrijden (Uitslagen) ophalen

We roepen het `TeamManage` endpoint aan met commando `GETWEDS` om de volledige wedstrijdkalender van deze ploeg op te halen, en filteren op de reeds gespeelde wedstrijden (de uitslagen).

In [ ]:
def get_team_played_matches(team_guid: str) -> pd.DataFrame:
    """Haalt alle gespeelde wedstrijden (tab uitslagen) op voor een ploeg."""
    payload = {
        "AuthHeader": "na",
        "WQVer": "nov2",
        "Command": "GETWEDS",
        "TeamGUID": team_guid,
        "CRUD": "R",
    }
    resp = requests.put(f"{VBL_BASE_URL}/TeamManage", headers=API_HEADERS, json=payload, timeout=15)
    resp.raise_for_status()
    all_matches = resp.json().get("wedList", [])
    
    # Filter op gespeelde wedstrijden (gespeeld == 'G' of waar een uitslag voor gekend is)
    played = [m for m in all_matches if m.get("uitslag") and str(m.get("uitslag")).strip()]
    df = pd.DataFrame(played)
    
    # Selecteer en hernoem de meest relevante kolommen
    cols = ["guid", "datumString", "beginTijd", "tTNaam", "tUNaam", "uitslag", "pouleNaam"]
    return df[[c for c in cols if c in df.columns]].rename(columns={
        "guid": "wedguid",
        "datumString": "datum",
        "beginTijd": "uur",
        "tTNaam": "thuisploeg",
        "tUNaam": "uitploeg",
        "pouleNaam": "reeks",
    })

df_played = get_team_played_matches(SELECTED_TEAM_GUID)
print(f"Aantal gespeelde wedstrijden: {len(df_played)}")
display(df_played)

## Stap 4 & 5: Wedstrijdverslagen ophalen via DWF (Matchdetailpdf)

Wanneer je op VBL Web op een wedstrijd klikt en het **uitgebreid verslag** (`Matchdetailpdf`) opent, haalt de webapp twee cruciale databronnen op:
1. `DwfDeelByWedGuid`: De deelnemers (spelers, rugnummers, coaches) van beide teams (`TtDeel` en `TuDeel`).
2. `DwfVgngByWedGuid`: De chronologische wedstrijdgebeurtenissen (`GebNis`), waarbij `GebType == 30` alle fouten voorstelt.

We bouwen een parser die deze twee combineert, zodat we elke fout rechtstreeks koppelen aan de speler, diens rugnummer en diens team.

In [ ]:
def parse_match_fouls(wedguid: str) -> list:
    """Haalt deelnemers en fouten op voor ??n wedstrijd-GUID."""
    payload = {
        "AuthHeader": "na",
        "WQVer": "nov2",
        "WedGUID": wedguid,
        "CRUD": "R",
    }
    
    try:
        r_deel = requests.put(f"{VBL_BASE_URL}/DwfDeelByWedGuid", headers=API_HEADERS, json=payload, timeout=10)
        r_vgng = requests.put(f"{VBL_BASE_URL}/DwfVgngByWedGuid", headers=API_HEADERS, json=payload, timeout=10)
    except Exception as e:
        print(f"Fout bij ophalen van {wedguid}: {e}")
        return []
        
    if r_deel.status_code != 200 or r_vgng.status_code != 200:
        return []
        
    data_deel = r_deel.json()
    data_vgng = r_vgng.json()
    
    # Bouw opzoekwoordenboek van spelers op basis van RelGUID
    players = {}
    for p in data_deel.get("TtDeel", []):
        players[p.get("RelGUID")] = {
            "naam": p.get("Naam"),
            "rugnr": p.get("RugNr"),
            "team_kant": "Thuis",
            "functie": p.get("Functie"),
        }
    for p in data_deel.get("TuDeel", []):
        players[p.get("RelGUID")] = {
            "naam": p.get("Naam"),
            "rugnr": p.get("RugNr"),
            "team_kant": "Uit",
            "functie": p.get("Functie"),
        }
        
    # Doorloop alle gebeurtenissen en filter op fouten (GebType == 30)
    foul_records = []
    for g in data_vgng.get("GebNis", []):
        if g.get("GebType") == 30:  # 30 = Fout
            rel = g.get("RelGUID")
            p_info = players.get(rel, {
                "naam": f"Onbekend / Bank ({rel})",
                "rugnr": g.get("RugNr", ""),
                "team_kant": "Thuis" if g.get("TofU") == "T" else "Uit",
                "functie": "S",
            })
            
            raw_code = str(g.get("Text", "")).strip()
            
            # Bepaal hoofdcategorie van de fout
            # P: Persoonlijk, T: Technisch, U: Onsportief, D: Diskwalificerend, C/B: Coach/Bank
            if raw_code.startswith("T") or raw_code.startswith("C") or raw_code.startswith("B"):
                cat = "Technisch (T)"
            elif raw_code.startswith("U"):
                cat = "Onsportief (U)"
            elif raw_code.startswith("D"):
                cat = "Diskwalificerend (D)"
            else:
                cat = "Persoonlijk (P)"
                
            foul_records.append({
                "wedguid": wedguid,
                "RelGUID": rel,
                "Naam": p_info["naam"],
                "RugNr": p_info["rugnr"],
                "TeamKant": p_info["team_kant"],
                "FoutType": raw_code,
                "FoutCategorie": cat,
                "Periode": g.get("Periode"),
                "Minuut": g.get("Minuut"),
            })
            
    return foul_records

print("Functie parse_match_fouls gereed.")

### Test op ??n wedstrijd
We testen de parser eerst op de voorbeeldfiche uit de vraag (`BVBL26279100LAHSE21AAN`).

In [ ]:
TEST_WEDGUID = "BVBL26279100LAHSE21AAN"
test_fouls = parse_match_fouls(TEST_WEDGUID)
df_test = pd.DataFrame(test_fouls)

print(f"Totaal aantal fouten in testwedstrijd: {len(df_test)}")
display(df_test.head(10))

print("\nPivot tabel van de testwedstrijd (Aantal fouten per speler per type):")
display(pd.crosstab(
    index=[df_test["TeamKant"], df_test["Naam"], df_test["RugNr"]],
    columns=df_test["FoutType"],
    margins=True,
    margins_name="Totaal"
))

## Stap 6: Alle voorbije wedstrijden doorlopen en fouten per speler aggregeren

Nu doorlopen we **alle gespeelde wedstrijden** van de geselecteerde ploeg. We verzamelen alle fouten en bouwen een overzichtstabel per speler van onze club.

In [ ]:
all_season_fouls = []

print(f"Bezig met verwerken van {len(df_played)} wedstrijden voor {selected_team_name}...")
for idx, row in df_played.iterrows():
    w_guid = row["wedguid"]
    w_datum = row["datum"]
    w_thuis = row["thuisploeg"]
    w_uit = row["uitploeg"]
    
    # Bepaal of onze ploeg thuis of uit speelde
    is_own_home = (SELECTED_TEAM_GUID in row.get("tTGUID", "")) or ("Haantjes" in w_thuis)
    own_side = "Thuis" if is_own_home else "Uit"
    
    fouls = parse_match_fouls(w_guid)
    for f in fouls:
        f["datum"] = w_datum
        f["match_label"] = f"{w_thuis} vs {w_uit}"
        f["is_eigen_speler"] = (f["TeamKant"] == own_side)
        all_season_fouls.append(f)

df_all_fouls = pd.DataFrame(all_season_fouls)
print(f"Klaar! Totaal geregistreerde fouten: {len(df_all_fouls)}")

### Volledig Overzicht: Fouten per speler van de eigen ploeg

We filteren op de spelers van de eigen club en maken een overzichtelijke matrix met:
- Persoonlijke fouten (`P`, `P1`, `P2`, `P3`)
- **Technische fouten** (`T`, `T1`, `C`, `B`)
- **Onsportieve fouten** (`U`, `U1`, `U2`)
- **Diskwalificerende fouten** (`D`, `DI`, `GD`)
- Totaal aantal fouten

In [ ]:
# Filter enkel op spelers van onze eigen ploeg
df_own_fouls = df_all_fouls[df_all_fouls["is_eigen_speler"]].copy()

# Pivot tabel per speler x fouttype
pivot_player_fouls = pd.crosstab(
    index=df_own_fouls["Naam"],
    columns=df_own_fouls["FoutType"],
    margins=False
)

# Bereken totalen per hoofdcategorie
p_cols = [c for c in pivot_player_fouls.columns if c.startswith("P")]
t_cols = [c for c in pivot_player_fouls.columns if c.startswith("T") or c.startswith("C") or c.startswith("B")]
u_cols = [c for c in pivot_player_fouls.columns if c.startswith("U")]
d_cols = [c for c in pivot_player_fouls.columns if c.startswith("D")]

summary = pivot_player_fouls.copy()
summary["Totaal_P"] = summary[p_cols].sum(axis=1) if p_cols else 0
summary["Totaal_T (Technisch)"] = summary[t_cols].sum(axis=1) if t_cols else 0
summary["Totaal_U (Onsportief)"] = summary[u_cols].sum(axis=1) if u_cols else 0
summary["Totaal_D (Diskwalificerend)"] = summary[d_cols].sum(axis=1) if d_cols else 0
summary["TOTAAL"] = summary["Totaal_P"] + summary["Totaal_T (Technisch)"] + summary["Totaal_U (Onsportief)"] + summary["Totaal_D (Diskwalificerend)"]

# Sorteer eerst op Technische fouten, dan Onsportieve fouten, dan Totaal
summary = summary.sort_values(by=["Totaal_T (Technisch)", "Totaal_U (Onsportief)", "TOTAAL"], ascending=False)

print(f"Foutenoverzicht voor {selected_team_name}:")
display(summary)

### Detail van Technische en Onsportieve fouten
Hier lichten we specifiek alle **technische**, **onsportieve** en **diskwalificerende** fouten uit, inclusief de datum, wedstrijd, periode en minuut waarop ze zijn opgelopen.

In [ ]:
speciale_fouten = df_own_fouls[df_own_fouls["FoutCategorie"].isin([
    "Technisch (T)", "Onsportief (U)", "Diskwalificerend (D)"
])].sort_values(by=["FoutCategorie", "Naam", "datum"])

if speciale_fouten.empty:
    print("Geen technische of onsportieve fouten geregistreerd voor deze ploeg tot nu toe! Fair play!")
else:
    print(f"Aantal speciale fouten (T/U/D): {len(speciale_fouten)}")
    display(speciale_fouten[["datum", "Naam", "RugNr", "FoutType", "FoutCategorie", "Periode", "Minuut", "match_label"]])

## Stap 7: Visualisatie

We visualiseren de verdeling van fouten over de spelers en wedstrijden.

In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    top_players = summary.head(12)
    
    # Gestapelde staafdiagram per categorie
    bars_p = ax.bar(top_players.index, top_players["Totaal_P"], label="Persoonlijk (P)", color="#1f77b4")
    bars_u = ax.bar(top_players.index, top_players["Totaal_U (Onsportief)"], bottom=top_players["Totaal_P"], label="Onsportief (U)", color="#ff7f0e")
    bars_t = ax.bar(top_players.index, top_players["Totaal_T (Technisch)"], bottom=top_players["Totaal_P"] + top_players["Totaal_U (Onsportief)"], label="Technisch (T)", color="#d62728")
    
    ax.set_title(f"Foutenverdeling per speler ? {selected_team_name}", fontsize=14, fontweight="bold", pad=15)
    ax.set_ylabel("Aantal fouten", fontsize=12)
    ax.set_xlabel("Speler", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Stap 8: Detailverslag van ??n specifieke wedstrijd (zoals op Matchdetailpdf)

Hieronder kan je voor eender welke wedstrijd-GUID het volledige wedstrijdblad oproepen en de foutenlijst van beide ploegen inspecteren.

In [ ]:
def toon_uitgebreid_verslag(wedguid: str):
    """Toont het wedstrijdblad en alle fouten van beide ploegen voor een specifieke wedstrijd."""
    fouls = parse_match_fouls(wedguid)
    if not fouls:
        print(f"Geen verslag of fouten gevonden voor {wedguid}")
        return
        
    df = pd.DataFrame(fouls)
    print(f"=== Wedstrijdverslag voor {wedguid} ===")
    
    for team_kant in ["Thuis", "Uit"]:
        team_f = df[df["TeamKant"] == team_kant]
        print(f"\n--- {team_kant}ploeg ({len(team_f)} fouten) ---")
        pivot = pd.crosstab(
            index=[team_f["Naam"], team_f["RugNr"]],
            columns=team_f["FoutType"],
            margins=True,
            margins_name="Totaal"
        )
        display(pivot)

# Voorbeeld met de match uit de vraag:
toon_uitgebreid_verslag("BVBL26279100LAHSE21AAN")

## Samenvatting

### Q&A
- **Hoe worden de wedstrijden en fouten opgehaald?**
  Via de offici?le VBL / Wisseq WebService (`https://vblcb.wisseq.eu/VBLCB_WebService/data`):
  - `OrgMatchesByGuid`: leest de club- en ploeg-GUIDs.
  - `TeamManage (GETWEDS)`: haalt de gespeelde wedstrijden (uitslagen) op.
  - `DwfDeelByWedGuid`: haalt de deelnemers en rugnummers op van het digitale wedstrijdblad (DWF).
  - `DwfVgngByWedGuid`: haalt alle wedstrijdgebeurtenissen op, waarbij `GebType 30` de fouten voorstelt met het exacte fouttype (`P`, `P1`, `P2`, `P3`, `T`, `T1`, `U`, `D`, `DI`, etc.).

### Data Analysis Key Findings
- De API levert real-time toegang tot het offici?le wedstrijdblad zonder dat authenticatie vereist is (`authorization: 'na'`, `ssoBearer: 'ssoWQAndre'`).
- Zowel persoonlijke als technische, onsportieve en diskwalificerende fouten worden geregistreerd per speler, inclusief periode en minuut.